# Savant 02, reports on real data

This notebook needs no GPU. It cleans each file in the private savant-real dataset and writes one HTML report per file with ranked insights, charts and the cleaning report. Long tables such as the CDC indicators are analyzed one measure at a time. The reports are in the output tab.

In [ ]:
import glob
import subprocess
import sys
import time
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel], check=True)
FILES = sorted(
    p
    for p in glob.glob("/kaggle/input/**/savant-real/**/*", recursive=True)
    if p.lower().endswith((".csv", ".tsv", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"))
)
print(len(FILES), "files")

In [ ]:
from savant.clean import clean_file
from savant.engine import Settings, analyze
from savant.insights import insights_text, rank
from savant.longform import analyze_long, detect
from savant.report_html import write_report

for path in FILES:
    name = Path(path).stem
    t0 = time.time()
    try:
        df, cleaning = clean_file(path)
        write_report(df, f"/kaggle/working/{name}.html", title=name, cleaning=cleaning.to_text())
        layout = detect(df)
        print(f"===== {name}, {df.height} rows, report written")
        if layout is not None:
            print(f"long table: {layout.describe()}")
            for measure, res in analyze_long(df, Settings()).items():
                ranked = rank(res.findings, limit=4, time_column=res.card.time_column)
                print(f"-- {measure}\n{insights_text(ranked)}")
        else:
            res = analyze(df, Settings())
            print(insights_text(rank(res.findings, time_column=res.card.time_column)))
    except Exception as e:  # one unreadable file should not stop the others
        print(f"===== {name} failed: {type(e).__name__}: {e}")
    print(f"{time.time() - t0:.0f}s\n")